In [ ]:
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import pygris

import geopandas as gpd
import pickle 

pd.set_option('display.max_rows', 50)
pd.options.mode.chained_assignment = None


In [ ]:
def convert_tract_to_fips(tract_id):
    tract_str = str(tract_id).strip()

    if "." in tract_str:
        before_decimal, after_decimal = tract_str.split(".")
        before_decimal = before_decimal.zfill(4)
        after_decimal = after_decimal.ljust(2, "0")
    else:
        before_decimal = tract_str.zfill(4)
        after_decimal = "00"

    # THIS IS 'PROPER', REMOVING THE LEADING FIPS CODE TO JOIN WITH OTHER DATASETS
    # return f"39035{before_decimal}{after_decimal}"
    return f"{before_decimal}{after_decimal}"

In [ ]:
tract_cols = [
    'NAME',
    'ALAND',
    'AWATER',
    'geometry',
]

# Downloading 2010 and 2020 tracts for Cuyahoga County, Ohio.
tracts_2010 = pygris.tracts(state="39", county="035", year=2015)[tract_cols]
tracts_2020 = pygris.tracts(state="39", county="035", year=2020)[tract_cols]

tracts_2010['NAME'] = tracts_2010['NAME'].astype(str).apply(convert_tract_to_fips)
tracts_2020['NAME'] = tracts_2020['NAME'].astype(str).apply(convert_tract_to_fips)

# Excluding tract 9900, which represents a water geometry over lake erie
tracts_2010 = tracts_2010[tracts_2010['NAME'] != '9900']
tracts_2020 = tracts_2020[tracts_2020['NAME'] != '9900']

There are 381 tract names in 2010 that match with 2020. However, there are over 60 tracts that are not accounted for within this. Additionally within the 381 tracts that do match in names, there are some that have grown in area.

In [ ]:
count_overlap = 0
for i in tracts_2010["NAME"].unique():
    if i in tracts_2020["NAME"].unique():
        count_overlap += 1
count_overlap

In [ ]:
tracts_allign_df = pd.DataFrame(columns=['NAME_10','NAME_20','OL_PROP'])

# Function handling the calculation of the overlap area
ol_area_fn = lambda intersect_row: (row.geometry.intersection(intersect_row.geometry).area / row.geometry.area)

for i in range(tracts_2020.shape[0]):
    row = tracts_2020.iloc[i]
    intersect = tracts_2010[tracts_2010.geometry.intersects(row.geometry)]

    intersect['NAME_20'] = row['NAME']
    intersect['OL_PROP'] = intersect.apply(ol_area_fn, axis=1)
    intersect = intersect.drop(columns=['geometry','ALAND','AWATER']).rename(columns={'NAME': 'NAME_10'})
    intersect = intersect[intersect['OL_PROP'] > 0.08]

    tracts_allign_df = pd.concat([tracts_allign_df, intersect])


# TODO handle this special case, it is a tract that is fully within another tract and is absorbed (merged)
# Adding a found special case to the dataframe
# tracts_allign_df.loc[len(tracts_allign_df)] = ['1948', '1971', 1.0]

In [ ]:
close_allign_df = tracts_allign_df[tracts_allign_df['OL_PROP'] > 0.9]

In [ ]:
# Get rows with name_10 duplicates
dupe_drop_idx = close_allign_df[close_allign_df.duplicated('NAME_10', keep=False)].index
# Drop indexes
close_allign_df = close_allign_df.drop(dupe_drop_idx)

In [ ]:
print('all','10','20')
close_allign_df.shape, close_allign_df['NAME_10'].unique().shape, close_allign_df['NAME_20'].unique().shape,

In [ ]:
# create a dict of 2010 to 2020 from name_10 and name_20
tracts_dict = dict(zip(close_allign_df['NAME_10'], close_allign_df['NAME_20']))
list(tracts_dict.items())[:10]

In [ ]:
# save as a pickle file
with open('data/pickle_files/close_enough.pickle', 'wb') as handle:
    pickle.dump(tracts_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
pd.set_option('display.max_columns', 200)

In [ ]:
census_data = pd.read_csv('data/ohio_census_data.csv')
census_data.describe()

In [ ]:
# nt_assign_df = tracts_allign_df.copy()
# # add '01' to name_10 and '02' to name_20
# nt_assign_df['NAME_10'] = nt_assign_df['NAME_10'].apply(lambda x: x + '01')
# nt_assign_df['NAME_20'] = nt_assign_df['NAME_20'].apply(lambda x: x + '02')
# nt_assign_df.reset_index(drop=True, inplace=True)
# nt_assign_df['NTID'] = nt_assign_df.index

# nt_assign_df = pd.concat([nt_assign_df[['NAME_10', 'NTID']].rename(columns={'NAME_10': 'NAME'}), 
#                                      nt_assign_df[['NAME_20', 'NTID']].rename(columns={'NAME_20': 'NAME'})], 
#                                     axis=0).reset_index(drop=True)
# nt_assign_df.sort_values(by='NTID')
# # read 'data\ohio_census_data.csv'
# census_data = pd.read_csv('data/ohio_census_data.csv')[['NAME','tract','year']]

# census_data['tract'] = census_data.apply(lambda x: str(x['tract']) + '01' if x['year'] < 2020 else str(x['tract']) + '02', axis=1)
# census_data['NTID'] = ''

# census_data.head()

In [ ]:
# Set ntid to the index

In [ ]:
# Count the number of ol_prop with imperfect allignment but less than 2 % difference
tracts_allign_df[(tracts_allign_df['OL_PROP'] > 0.98) & (tracts_allign_df['OL_PROP'] < 0.999)].shape[0]

In [ ]:
# Generate a histogram of olprop for values > 0.9 but less than 1.0
plt.hist(tracts_allign_df[(tracts_allign_df['OL_PROP'] > 0.9) & (tracts_allign_df['OL_PROP'] < .999)]['OL_PROP'], bins=50)

In [ ]:
tracts_allign_df['NAME_10'].value_counts().head(50)

In [ ]:
# Get number of unique 2010 and 2020 tracts
num_2010 = len(tracts_allign_df['NAME_10'].unique())
num_2020 = len(tracts_allign_df['NAME_20'].unique())

print(num_2010,num_2020)
print(tracts_2010['NAME'].unique().shape[0],tracts_2020['NAME'].unique().shape[0])

In [ ]:
tracts_2010_that_dont_merge = tracts_allign_df['NAME_10'].value_counts()
lst_tracts_2010_good = list(tracts_2010_that_dont_merge[tracts_2010_that_dont_merge == 1].index)

In [ ]:
tracts_2020_that_dont_merge = tracts_allign_df['NAME_20'].value_counts()
lst_tracts_2020_good = list(tracts_2020_that_dont_merge[tracts_2020_that_dont_merge == 1].index)

In [ ]:
tracts_2010.head()

In [ ]:
cleaned_tracts_2010 = tracts_2010[tracts_2010['NAME'].isin(lst_tracts_2010_good)]
cleaned_tracts_2020 = tracts_2020[tracts_2020['NAME'].isin(lst_tracts_2020_good)]

In [ ]:
tracts_allign_df['NAME_10'].value_counts().head(20)

In [ ]:
changes_df = pd.read_csv("https://www2.census.gov/geo/docs/maps-data/data/rel2020/tract/tab20_tract20_tract10_natl.txt", delimiter="|", dtype=str)
changes_df = changes_df[changes_df['GEOID_TRACT_20'].str.startswith("39035")]
changes_df.head()

In [ ]:
# Add a column to changes_df representing the absolute difference between arealand_tract 10 and arealand_tract 20
changes_df['DIFF_ALAND'] = changes_df['AREALAND_TRACT_10'].astype(float) - changes_df['AREALAND_TRACT_20'].astype(float)
changes_df['DIFF_ALAND_ABS'] = changes_df['DIFF_ALAND'].abs()
changes_df['DIFF_ALAND_SIGN'] = np.sign(changes_df['DIFF_ALAND'])

In [ ]:
29866875.0	- 29834670.0	

In [ ]:
# Print changes_df sorted by DIFF_ALAND_ABS in descending order
changes_df.sort_values(by='DIFF_ALAND_ABS', ascending=False)

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
changes_df.sort_values(by='DIFF_ALAND', ascending=False).head(20)

In [ ]:
# We will create a function that creates a dicts that consists of tract ids from 2020 as keys and 2010 tract ids as values. 
# It will only consists of tracts that do NOT consist from merging or splitting. It has to be the exact same tract. 
# The AREALAND_TRACT_20 and AREALAND_TRACT_10 are the exact same 
good_tract_ids = {}
for i in range(changes_df.shape[0]):
    row = changes_df.iloc[i]
    if row["AREALAND_TRACT_20"] == row["AREALAND_TRACT_10"]:
        if row["GEOID_TRACT_20"] != row["GEOID_TRACT_10"]:
            print(f"THIS ID IS WEIRD:")
        good_tract_ids[row["GEOID_TRACT_20"]] = row["GEOID_TRACT_10"]

In [ ]:
# save as a pickle file
with open('data/pickle_files/good_tract_ids.pickle', 'wb') as handle:
    pickle.dump(good_tract_ids, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# load pickle files, count unique keys and values in each
with open('data/pickle_files/close_enough.pickle', 'rb') as handle:
    close_enough = pickle.load(handle)
with open('data/pickle_files/good_tract_ids.pickle', 'rb') as handle:
    good_tract_ids = pickle.load(handle)
print(f"close_enough: {len(close_enough.keys())} keys, {len(close_enough.values())} values")
print(f"good_tract_ids: {len(good_tract_ids.keys())} keys, {len(good_tract_ids.values())} values")
